# Frunze bolnave pentru lumina de zi (model_frunze_bolnave_v3) — maxim ~1 oră

**De ce:** pe EscaYard, parcela B7 (pozată **seara**) modelul găsește ~6 frunze bolnave pe un butuc bolnav, dar pe B9
(pozată **la amiază**) doar ~1,9, așa că pe B9 găsește doar 3 din 12 butuci bolnavi. Modelul a fost antrenat pe poze
cu **blitz**; în soare puternic simptomele „dispar”.

**Ce face:** pornește de la modelul actual și îl reantrenează pe **aceleași** poze FD (aceeași împărțire), cu augmentare
puternică de lumină: mai luminos, mai mult contrast, gamma, umbre (soare de amiază) + compresie JPEG și zgomot (ESP32).
Epoca se alege pe butucii **B7**; **B9** e doar testul final (vechi vs nou).

## Ce faceți voi (totul la început)
1. **Runtime → Change runtime type → T4 GPU → Save**, apoi **Runtime → Run all**.
2. Celula 3: „Connect to Google Drive”.
3. Celula 4: dacă `model_frunze_bolnave.pt` nu e în Drive, alegeți-l de pe laptop (`models/`).

Durată estimată: **~40–55 min** (descărcări ~10 min, antrenare ~20 min, alegere + test ~10 min).
Rezultatul: `MyDrive/vie_hackathon/leaf_v3_runs/model_frunze_bolnave_v3.pt` + tabelul vechi vs nou.

## 1. Placa video

In [ ]:
!nvidia-smi

## 2. Instalare

In [ ]:
!pip install -q ultralytics==8.4.163 py7zr
import ultralytics, albumentations
print("ultralytics", ultralytics.__version__, "| albumentations", albumentations.__version__)

## 3. Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/vie_hackathon'
R = f"{OUT}/leaf_v3_runs"
os.makedirs(R, exist_ok=True)
os.environ["LEAF_ROOT"] = "/content/lv"
os.environ["LEAF_RUNS"] = R
for d in ("data/escayard/photos", "models"):
    os.makedirs(f"/content/lv/{d}", exist_ok=True)

## 4. Modelul actual de frunze (punctul de plecare)

In [ ]:
import shutil
from google.colab import files

if os.path.exists(f"{OUT}/model_frunze_bolnave.pt"):
    shutil.copy(f"{OUT}/model_frunze_bolnave.pt", "/content/lv/models/model_frunze_bolnave.pt")
while not os.path.exists("/content/lv/models/model_frunze_bolnave.pt"):
    print("model_frunze_bolnave.pt is not in Drive -> choose it from the laptop (folder models/)")
    for name, data in files.upload().items():
        if "model_frunze_bolnave" in name and "v2" not in name:
            open("/content/lv/models/model_frunze_bolnave.pt", "wb").write(data)
print("OK: model_frunze_bolnave.pt")

## 5. EscaYard (1,4 GB) se descarcă în fundal cât timp luăm pozele FD

In [ ]:
import subprocess
escayard = subprocess.Popen(["bash", "-c", '''
cd /content
curl -sSfL --retry 3 -o /content/lv/data/escayard/datasheet.csv "https://zenodo.org/api/records/10362568/files/datasheet.csv/content" &&
curl -sSfL --retry 3 -o escayard.7z "https://zenodo.org/api/records/10362568/files/PLANTs_SmartPhonePhotos_JPG.7z/content" &&
python3 -c "import py7zr; py7zr.SevenZipFile('escayard.7z').extractall('/content/lv/data/escayard/photos')" &&
echo ESCAYARD_OK'''], stdout=open("/content/dl_escayard.log", "w"), stderr=subprocess.STDOUT)
print("EscaYard download started")

## 6. Pozele FD (Bordeaux) — aceeași descărcare ca la primul model de frunze (~1 GB)

In [ ]:
import base64, collections, json, re, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

API = "https://data.mendeley.com/public-api/datasets/3dr9r3w3jn"
FOLDERS = {
    "complete_CS20": "cc48eedb-c553-40cc-8d10-409cd5115763",
    "complete_UB20": "d805e05d-e696-4fc5-9256-6eef0f22b529",
    "soft_CF21": "9883150a-0a0a-4e18-a37d-d9dd1b33cd5a",
    "soft_CS21": "82f6b951-8daa-4fd8-85ef-0cc9e24fdc6d",
    "soft_M21": "fbdd115d-f7ed-4494-a14e-71d58d246165",
    "soft_SB21": "8c069f41-c13b-464a-a1c4-c1b0a962fcaa",
    "soft_UB20": "74f8f93f-762d-4f23-b28b-b0c24a9bd66d",
    "soft_UB21": "90153e4b-e3f1-43a4-99b6-0119eff52a0f",
}
HEADERS = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}
LEAF = {"FD leaf", "ESCA leaf", "confounding leaf", "facteur confondant +"}

ALL = Path("/content/fd_all")
(ALL / "images").mkdir(parents=True, exist_ok=True)
(ALL / "labels").mkdir(parents=True, exist_ok=True)

def get(url, as_json=True):
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=120) as r:
                data = r.read()
            return json.loads(data) if as_json else data
        except Exception as e:
            err = e
    raise err

jobs, jpg_urls = [], {}
for folder, fid in FOLDERS.items():
    for f in get(f"{API}/files?folder_id={fid}&version=2"):
        url = f["content_details"]["download_url"]
        if f["filename"].endswith(".json"):
            stem = re.sub(r"\s*\((\d+)\)", r"_dup\1", Path(f["filename"]).stem)  # „im_00031 (2)” → „im_00031_dup2”
            stem = re.sub(r"[^A-Za-z0-9_]", "", f"{folder}_{stem}")
            jobs.append((folder, stem, url))
        else:
            jpg_urls[(folder, f["filename"])] = url
print(len(jobs), "fișiere de adnotare")

def convert(job):
    folder, stem, url = job
    img_path, lbl_path = ALL / "images" / f"{stem}.jpg", ALL / "labels" / f"{stem}.txt"
    if img_path.exists() and lbl_path.exists():
        return None
    a = get(url)
    W, H = a["imageWidth"], a["imageHeight"]
    lines, kept, skipped = [], collections.Counter(), collections.Counter()
    for s in a["shapes"]:
        if s["shape_type"] == "rectangle" and s["label"] in LEAF:
            (x1, y1), (x2, y2) = s["points"]
            x1, x2 = sorted((min(max(x1, 0), W), min(max(x2, 0), W)))
            y1, y2 = sorted((min(max(y1, 0), H), min(max(y2, 0), H)))
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            lines.append(f"0 {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
            kept[s["label"]] += 1
        else:
            skipped[s["label"]] += 1
    if a.get("imageData"):
        img = base64.b64decode(a["imageData"])
    else:
        img = get(jpg_urls[(folder, a["imagePath"])], as_json=False)
    img_path.write_bytes(img)
    lbl_path.write_text("\n".join(lines))
    return kept, skipped

kept, skipped = collections.Counter(), collections.Counter()
with ThreadPoolExecutor(16) as ex:
    for i, res in enumerate(ex.map(convert, jobs), 1):
        if res:
            kept.update(res[0])
            skipped.update(res[1])
        if i % 25 == 0 or i == len(jobs):
            print(f"Descărcat {i}/{len(jobs)} ({i / len(jobs):.0%})", flush=True)

print("Poze:", len(list((ALL / "images").glob("*.jpg"))))
print("Frunze bolnave păstrate:", dict(kept), "— total", sum(kept.values()))
print("Ignorate:", dict(skipped))

## 7. Aceeași împărțire train / valid / test ca la primul model (seed 0)

In [ ]:
import random, shutil, yaml

DST = Path("/content/fd-vie")
if DST.exists():
    shutil.rmtree(DST)

groups = collections.defaultdict(list)
for img in sorted((ALL / "images").glob("*.jpg")):
    groups[re.sub(r"_dup\d+$", "", img.stem)].append(img)

keys = sorted(groups)
random.seed(0)
random.shuffle(keys)
n = len(keys)
split_of = {k: ("train" if i < 0.70 * n else "valid" if i < 0.85 * n else "test") for i, k in enumerate(keys)}

counts = collections.Counter()
for k, imgs in groups.items():
    split = split_of[k]
    for img in imgs:
        for sub in ("images", "labels"):
            (DST / split / sub).mkdir(parents=True, exist_ok=True)
        shutil.copy(img, DST / split / "images" / img.name)
        shutil.copy(ALL / "labels" / f"{img.stem}.txt", DST / split / "labels" / f"{img.stem}.txt")
        counts[split] += 1
print(dict(counts))

yaml.safe_dump({"path": str(DST), "train": "train/images", "val": "valid/images", "test": "test/images",
                "names": {0: "frunza_bolnava"}},
               open(DST / "data.yaml", "w"), allow_unicode=True)
print(open(DST / "data.yaml").read())

## 8. Așteptăm EscaYard

In [ ]:
import glob, time
while escayard.poll() is None:
    time.sleep(10)
print(open("/content/dl_escayard.log").read()[-300:])
print("EscaYard photos:", len(glob.glob("/content/lv/data/escayard/photos/**/*.jpg", recursive=True)))

## 9. Scriptul (același ca în proiect: `tools/leaf_daylight.py`)

In [ ]:
%%writefile /content/leaf_daylight.py
"""Diseased-leaf model for daylight (model_frunze_bolnave_v3): fine-tuned from model_frunze_bolnave.pt with strong light
augmentation, because the model was trained on flash photos (Bordeaux FD dataset) and loses the symptoms in harsh sun.

Measured on 27 Sept (tools/tune_vine_disease.py): EscaYard B7 was photographed in the evening, B9 at midday. Diseased
leaves found per sick vine: 6.1 in B7, 1.9 in B9 -> B9 only 3/12 sick vines found. No rule or tiling fixed it.
Auto-labels on EscaYard are not usable (almost every photo has leaves at an unsure 0.25-0.5), so the only new
ingredient is the augmentation: brighter / higher-contrast / gamma / shadows (midday sun) + JPEG and noise (ESP32).

Data: the same FD train / valid / test split as the first leaf model (colab/fd_frunze_bolnave.ipynb, seed 0).
Epoch chosen on EscaYard B7 at vine level (today's rule: at least 3 leaves at conf >= 0.5); B9 is only the final test.

Steps (the same file runs in Colab: LEAF_ROOT = folder with data/ and models/, LEAF_RUNS = where runs go):
  python leaf_daylight.py train --data /content/fd-vie/data.yaml   # ~15-20 min on a T4
  python leaf_daylight.py select --run leaf_v3                     # every 2nd epoch on B7 vines -> best epoch
  python leaf_daylight.py report --weights <best.pt>               # old vs new: B9 vines + FD test
"""
import argparse
import csv
import json
import os
import shutil
import time
from pathlib import Path

ROOT = Path(os.environ.get("LEAF_ROOT", Path(__file__).resolve().parent.parent))
RUNS = Path(os.environ.get("LEAF_RUNS", ROOT / "runs" / "leaf_v3"))
ESCA = ROOT / "data" / "escayard"
OLD = ROOT / "models" / "model_frunze_bolnave.pt"
IMGSZ, CONF, MIN_LEAVES = 960, 0.5, 3  # as in vineyard/config.py and the suspect-vine rule
MAX_AREA = 0.08  # vineyard/config.py MAX_BOX_AREA: a "leaf" bigger than 8% of the photo is soil or stones
MAX_SIDE = 2048  # EscaYard photos are shrunk like in tools/evaluate_escayard.py


def light_augmentations():
    """Midday sun + ESP32 camera. Each transform is skipped if this albumentations version does not know it."""
    try:
        import albumentations as A
    except ImportError:
        print("albumentations is not installed: training with the colour (HSV) augmentation only")
        return None
    wanted = [
        lambda: A.RandomBrightnessContrast(brightness_limit=(-0.1, 0.4), contrast_limit=(-0.2, 0.4), p=0.6),
        lambda: A.RandomGamma(gamma_limit=(60, 140), p=0.4),
        lambda: A.RandomShadow(p=0.2),
        lambda: A.CLAHE(p=0.1),
        lambda: A.ImageCompression(quality_range=(40, 90), p=0.3),
        lambda: A.GaussNoise(p=0.2),
        lambda: A.MotionBlur(p=0.1),
    ]
    out = []
    for make in wanted:
        try:
            t = make()
            A.to_dict(t)  # Ultralytics saves the transforms this way: one that cannot be saved would stop the training
            out.append(t)
        except Exception as e:  # older / newer albumentations with other argument names
            print("skipping an augmentation:", e)
    return out


# ---------- vine level (EscaYard) ----------

def vine_rows():
    rows = []
    with open(ESCA / "datasheet.csv", encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            status = r["Esca"].strip().upper()
            if r["File"].strip() and status in ("YES", "NO"):
                rows.append((r["File"].strip(), r["Vineyard"].strip(), status == "YES"))
    return rows


def leaves_per_photo(model, block):
    """{photo: [conf of each diseased leaf >= 0.2]} for the vines of one EscaYard block."""
    import cv2
    photos = {p.name: p for p in (ESCA / "photos").rglob("*.jpg")}
    out = {}
    for name, b, _ in vine_rows():
        if b != block or name not in photos:
            continue
        img = cv2.imread(str(photos[name]))
        h, w = img.shape[:2]
        s = MAX_SIDE / max(h, w)
        if s < 1:
            img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
            h, w = img.shape[:2]
        r = model.predict(img, imgsz=IMGSZ, conf=0.2, verbose=False)[0]
        out[name] = [c for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist())
                     if (x2 - x1) * (y2 - y1) <= MAX_AREA * w * h]
    return out


def vine_scores(leaves, block, conf=CONF, min_leaves=MIN_LEAVES):
    """(sick found, sick, healthy flagged, healthy) with the rule "at least min_leaves leaves at conf >= conf"."""
    tp = n_sick = fp = n_healthy = 0
    for name, b, sick in vine_rows():
        if b != block or name not in leaves:
            continue
        flagged = sum(c >= conf for c in leaves[name]) >= min_leaves
        if sick:
            n_sick, tp = n_sick + 1, tp + flagged
        else:
            n_healthy, fp = n_healthy + 1, fp + flagged
    return tp, n_sick, fp, n_healthy


def youden(s):
    """sick found share - healthy flagged share (1 = perfect, 0 = guessing)."""
    tp, n_sick, fp, n_healthy = s
    return (tp / n_sick if n_sick else 0) - (fp / n_healthy if n_healthy else 0)


# ---------- commands ----------

def train(args):
    from ultralytics import YOLO

    t0 = time.time()

    def print_epoch(trainer):
        ep, total = trainer.epoch + 1, trainer.epochs
        if ep > total:
            return
        elapsed = time.time() - t0
        print(f"==> Epoch {ep}/{total} ({ep / total:.0%}) | FD validation mAP50: "
              f"{trainer.metrics.get('metrics/mAP50(B)', 0.0):.3f} | {elapsed / 60:.0f} min | "
              f"left: max ~{elapsed / ep * (total - ep) / 60:.0f} min", flush=True)

    m = YOLO(args.init)
    m.add_callback("on_fit_epoch_end", print_epoch)
    extra = {}
    aug = light_augmentations()
    if aug:
        extra["augmentations"] = aug
    # AdamW with an explicit learning rate: optimizer="auto" picks its own and ignores lr0 on short fine-tunes
    m.train(data=args.data, imgsz=IMGSZ, epochs=args.epochs, patience=args.epochs, batch=args.batch,
            optimizer="AdamW", lr0=args.lr0, save_period=2, workers=2, cache="ram", project=str(RUNS),
            name=args.name, exist_ok=True, plots=False, hsv_s=0.7, hsv_v=0.7, fraction=args.fraction,
            amp=not args.no_amp, **extra)


def select(args):
    from ultralytics import YOLO
    w = RUNS / args.run / "weights"
    cands = sorted(w.glob("epoch*.pt"), key=lambda p: int(p.stem[5:])) + [w / "best.pt", w / "last.pt"]
    cands = [c for c in cands if c.exists()] + [OLD]
    table = []
    for i, c in enumerate(cands, 1):
        s = vine_scores(leaves_per_photo(YOLO(str(c)), "B7"), "B7")
        table.append((youden(s), c, s))
        print(f"{i}/{len(cands)} ({i / len(cands):.0%}) {c.name:26s} B7: sick found {s[0]}/{s[1]}, "
              f"healthy flagged {s[2]}/{s[3]}", flush=True)
    j, best, s = max(t for t in table if t[1] != OLD)
    old = next(t for t in table if t[1] == OLD)
    print(f"\nBest: {best.name} (B7 sick found {s[0]}/{s[1]}, healthy flagged {s[2]}/{s[3]}) | "
          f"old model: {old[2][0]}/{old[2][1]}, {old[2][2]}/{old[2][3]}")
    shutil.copy(best, RUNS / "model_frunze_bolnave_v3.pt")
    (RUNS / "leaf_v3_choice.json").write_text(json.dumps({"epoch": best.name, "b7": s, "b7_old": old[2]}))
    print("Saved:", RUNS / "model_frunze_bolnave_v3.pt")


def report(args):
    from ultralytics import YOLO
    rows = []
    for name, weights in (("old", OLD), ("new (v3)", Path(args.weights))):
        m = YOLO(str(weights))
        r = {"model": name}
        if args.data:
            res = m.val(data=args.data, split="test", imgsz=IMGSZ, verbose=False, plots=False)
            r["FD test mAP50 (flash photos)"] = f"{res.box.map50:.3f}"
        leaves = leaves_per_photo(m, "B9")
        s = vine_scores(leaves, "B9")
        r["B9 sick vines found (>=3 leaves)"] = f"{s[0]}/{s[1]}"
        r["B9 healthy vines flagged"] = f"{s[2]}/{s[3]}"
        sick = [len([c for c in leaves[n] if c >= CONF]) for n, b, e in vine_rows() if b == "B9" and e and n in leaves]
        r["B9 leaves per sick vine"] = f"{sum(sick) / max(1, len(sick)):.1f}"
        rows.append(r)
        print(f"{name} done", flush=True)
    keys = [k for k in rows[0] if k != "model"]
    print(f"\n{'':36s}{rows[0]['model']:>12s}{rows[1]['model']:>12s}")
    for k in keys:
        print(f"{k:36s}{rows[0][k]:>12s}{rows[1][k]:>12s}")
    (RUNS / "leaf_v3_report.json").write_text(json.dumps(rows, indent=1))


def main():
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest="cmd", required=True)
    t = sub.add_parser("train")
    t.add_argument("--data", required=True)
    t.add_argument("--name", default="leaf_v3")
    t.add_argument("--init", default=str(OLD))
    t.add_argument("--epochs", type=int, default=24)
    t.add_argument("--batch", type=int, default=8)
    t.add_argument("--lr0", type=float, default=0.0005)
    t.add_argument("--fraction", type=float, default=1.0)
    t.add_argument("--no-amp", action="store_true")
    s = sub.add_parser("select")
    s.add_argument("--run", default="leaf_v3")
    r = sub.add_parser("report")
    r.add_argument("--weights", required=True)
    r.add_argument("--data", help="FD data.yaml, to also score the FD test photos")
    args = p.parse_args()
    {"train": train, "select": select, "report": report}[args.cmd](args)


if __name__ == "__main__":
    main()

## 10. Antrenare (~20 min, 24 de epoci, progres pe epoci)

In [ ]:
!cd /content && python leaf_daylight.py train --data /content/fd-vie/data.yaml --epochs 24

## 11. Alegem epoca pe butucii B7 (~5 min)

In [ ]:
!cd /content && python leaf_daylight.py select --run leaf_v3

## 12. Testul final: vechi vs nou pe B9 (pozat la amiază) + testul FD (~5 min)

In [ ]:
!cd /content && python leaf_daylight.py report --weights "{R}/model_frunze_bolnave_v3.pt" --data /content/fd-vie/data.yaml

## 13. Pe laptop
Trimiteți-i lui Claude tabelul de mai sus. Dacă modelul nou e mai bun pe B9 fără să marcheze mai mulți butuci sănătoși,
descărcați `leaf_v3_runs/model_frunze_bolnave_v3.pt` în `models/`: îl verificăm pe laptop și pe tura 1 (alarme false
pe via sănătoasă de primăvară) înainte să-l punem în `vineyard/config.py`.